# Notebook: Twitter scrape inspection and completeness checklist

## Purpose
This notebook inspects scraped tweet CSV files, extracts basic metadata (row counts, created_at ranges, scrape timestamps, keywords), and produces per-file and per-keyword completeness reports for three analysis periods (Before Demo, Demo, After Demo).

## What it does
- Lists files in a configured data directory and prints small previews for CSV / JSON / text files.
- Scans `tweets_*.csv` files, counts tweets (chunked read), and computes min/max `created_at` dates.
- Classifies each file into duration buckets (Before Demo / Demo / After Demo / Outside Range).
- Checks per-file completeness against expected period windows and reports missing subranges to re-scrape.
- Aggregates by keyword and reports combined completeness and total tweet counts.
- Saves two CSV outputs:
    - `parsed_tweet_filenames_sorted.csv` — per-file metadata (sorted by scrape time)
    - `keyword_completeness_summary.csv` — per-keyword completeness summary

## Expectations / assumptions
- Tweet CSV filenames begin with `tweets_` and include a scrape timestamp suffix like `_YYYYMMDD_HHMMSS.csv`.
- Each tweet CSV contains a `created_at` column parseable by pandas (chunked reading used to be memory efficient).
- Date windows are hard-coded (2025):
    - Before Demo: 2025-08-01 — 2025-08-24
    - Demo: 2025-08-25 — 2025-09-08
    - After Demo: 2025-09-09 — 2025-09-30
- The notebook expects the file-listing cell to be run first (it defines `data_dir` and `items`).

## How to run
1. Run the file-listing cell (path detection tries a preferred Windows path and an alternate).
2. Run the parsing/analysis cell that builds `df` and writes the two CSV outputs.
3. Inspect saved CSVs or re-run after adjusting paths/periods.

## Configuration / tuning
- Adjust `data_dir` detection or set `data_dir` manually if files are elsewhere.
- Modify the hard-coded period dates at the top of the parsing cell to match your analysis windows.
- If `created_at` uses timezones or non-standard formats, update the date parsing parameters (the code tries a fallback if initial parse fails).
- Increase/decrease chunk size in `pd.read_csv(..., chunksize=...)` depending on memory.

## Notes and recommendations
- The script sums tweet counts across files to avoid newline-count errors inside tweet text.
- For large-scale re-scraping, add logging, retry/backoff, and checkpointing, and ensure compliance with Twitter terms and privacy considerations.
- If you need richer metadata or robust rate guarantees, consider using the official Twitter API v2 instead of scraping.


# TwitterScrapper — repository overview and usage guide https://github.com/DaBabyOx/TwitterScrapper

This repository (TwitterScrapper) provides scripts and utilities to collect tweets and related metadata from Twitter for analysis. Typical goals are building datasets, monitoring hashtags or accounts, and exporting tweet text, timestamps, user info, and other fields for downstream processing.

## What it likely contains
- One or more Python scripts that perform scraping/saving of tweets.
- A `requirements.txt` or `environment.yml` listing dependencies (e.g., `snscrape`, `tweepy`, `requests`, `pandas`).
- Example usage or CLI arguments (query, username, date ranges, output path).
- Data output in CSV/JSON and simple preview/processing helpers.

## How it generally works
- Accepts search parameters (keywords, hashtag, user, date window).
- Uses a scraping library or API client to fetch tweets and metadata.
- Saves results to disk (CSV/JSON) and may provide basic cleaning/previews.

## Quick setup
```bash
# clone repository
git clone https://github.com/DaBabyOx/TwitterScrapper.git
cd TwitterScrapper

# create virtual env and install deps
python -m venv .venv
source .venv/bin/activate   # macOS / Linux
.venv\Scripts\activate      # Windows
pip install -r requirements.txt
```

If the project uses the Twitter API, you will need API keys and to set them as environment variables or edit a config file as documented in the repo.

## Typical usage examples
```bash
# search by keyword and save to CSV
python scrape.py --query "covid19" --since 2023-01-01 --until 2023-02-01 --out tweets.csv

# scrape tweets from a user timeline
python scrape.py --user "elonmusk" --limit 1000 --out elon_tweets.json
```
(Check the repository README or script `--help` for exact options.)

## File/structure to look for
- README.md — usage details, examples
- scripts (e.g., `scrape.py`, `collect.py`) — main entry points
- `requirements.txt` — dependencies to install
- `config.example` or `.env.example` — where to place API keys
- sample output folder (CSV/JSON) or notebooks for analysis

## Recommendations and improvements
- Prefer `snscrape` for public tweet scraping without elevated API access; use Twitter API v2 if you need richer metadata or rate-guaranteed access.
- Add logging, retry/backoff, and checkpointing for large scrapes.
- Validate and document fields saved (id, text, created_at, user, lang, retweet_count, like_count).

## Legal, ethical, and practical notes
- Respect Twitter’s terms of service and rate limits. Scraping can violate policies; prefer official APIs when required.
- Consider users’ privacy when publishing or sharing scraped data.
- Large-scale scraping can cause IP rate-limiting; use polite delays and error handling.

If you want, I can open the repository and produce a precise README-style summary and exact run commands based on the actual files.

The scraped data needs to be checked.

In [ ]:
# List contents of a specified data folder and give quick previews for common text files
from pathlib import Path
import pandas as pd

# prefer the user-provided Windows path; try a forward-slash variant too, else fall back to existing data_dir or /data
preferred = Path(r"G:\My Drive\University Files\5th Semester\Data Mining\Twitter_Sentiment\data")
alt = Path("G:/My Drive/University Files/5th Semester/Data Mining/Twitter_Sentiment/data")

if preferred.exists():
    data_dir = preferred
elif alt.exists():
    data_dir = alt
else:
    try:
        data_dir  # use existing value if present in notebook
    except NameError:
        data_dir = Path("/data")
    print(f"Preferred paths not found; using {data_dir}")

print(f"Directory: {data_dir} (exists: {data_dir.exists()})\n")

items = sorted(data_dir.iterdir()) if data_dir.exists() else []
if not items:
    print("No files or folders found in", data_dir)
else:
    for p in items:
        t = "dir " if p.is_dir() else "file"
        size = f"{p.stat().st_size}" if p.is_file() else ""
        print(f"{t:4}  {p.name:40}  {size:>10}")

# update csv_files variable to reflect actual CSVs present
csv_files = sorted(data_dir.glob("*.csv")) if data_dir.exists() else []
print(f"\nFound {len(csv_files)} CSV files\n")

# Preview up to first 5 lines for CSV/text/json files
for p in items:
    if p.is_file() and p.suffix.lower() in {'.csv', '.txt', '.log', '.json'}:
        print(f"\n--- Preview: {p.name} ---")
        try:
            if p.suffix.lower() == '.csv':
                print(pd.read_csv(p, nrows=5).to_string(index=False))
            else:
                print(p.read_text(errors='replace')[:1000])
        except Exception as e:
            print("Could not preview file:", e)

Example of csv: tweets_Tolak_Tunjangan_since_2025_08__20251020_174858.csv



In [ ]:
import pandas as pd
import re
from datetime import date, timedelta

# --- !! MOVED DEFINITIONS HERE !! ---
# Define date ranges globally to be used by multiple functions
before_demo_start = date(2025, 8, 1)
before_demo_end = date(2025, 8, 24)
demo_start = date(2025, 8, 25)
demo_end = date(2025, 9, 8)
after_demo_start = date(2025, 9, 9)
after_demo_end = date(2025, 9, 30)

def get_duration_period(single_date):
    """Classifies a single date into Before Demo, Demo, or After Demo."""
    if single_date is None:
        return "N/A"
    
    # Note: These dates are now read from the global scope
    if before_demo_start <= single_date <= before_demo_end:
        return "Before Demo"
    elif demo_start <= single_date <= demo_end:
        return "Demo"
    elif after_demo_start <= single_date <= after_demo_end:
        return "After Demo"
    else:
        return "Outside Range"

# --- !! NEW HELPER FUNCTION FOR KEYWORD SUMMARY !! ---
def check_keyword_completeness(group, period_name, period_start, period_end, margin):
    """
    Checks the combined completeness for a keyword group against a specific period.
    """
    # Find all files in the group that have data overlapping with this period
    overlapping_files = group[
        (group['min_date_dt'].notna()) &
        (group['max_date_dt'].notna()) &
        (group['min_date_dt'].dt.date <= period_end) & 
        (group['max_date_dt'].dt.date >= period_start)
    ]

    if overlapping_files.empty:
        return {
            "Period": period_name,
            "Amount (Tweets)": 0, # <-- ADDED
            "Completeness": "No Data Found",
            "Combined Date Range": "N/A",
            "Missing to Scrape": f"Scrape: {period_start} to {period_end}"
        }

    # Find the earliest start date and latest end date from ALL files for this period
    overall_min_date = overlapping_files['min_date_dt'].min().date()
    overall_max_date = overlapping_files['max_date_dt'].max().date()
    
    # --- !! NEW !! Sum the tweets from all overlapping files ---
    total_tweets = overlapping_files['Amount (Tweets)'].sum()

    # Apply the same completeness logic as before, but to the *combined* dates
    completeness_check = "N/A"
    missing_data_to_scrape = "N/A"
    missing_parts = []

    is_complete = (overall_min_date <= period_start + margin) and (overall_max_date >= period_end - margin)
    completeness_check = "Complete" if is_complete else "Incomplete"

    if not is_complete:
        if overall_min_date > period_start + margin:
            missing_parts.append(f"since:{period_start} until:{overall_min_date - timedelta(days=1)}")
        if overall_max_date < period_end - margin:
            missing_parts.append(f"since:{overall_max_date + timedelta(days=1)} until:{period_end}")
    
    if missing_parts:
        missing_data_to_scrape = f"Scrape: {', '.join(missing_parts)}"

    return {
        "Period": period_name,
        "Amount (Tweets)": total_tweets, # <-- ADDED
        "Completeness": completeness_check,
        "Combined Date Range": f"{overall_min_date} to {overall_max_date}",
        "Missing to Scrape": missing_data_to_scrape
    }


# Regex to find the scrape date and time
scrape_datetime_re = re.compile(r'_(\d{8})_(\d{6})\.csv$')
# Regex to find the 'since' date
since_date_re = re.compile(r'^(\d{4})_(\d{2})(?:_(\d{2}))?')

parsed_data = []

# This code assumes the previous cell was run and the 'items' variable exists.
# 'items' is expected to be a list of Path objects from data_dir.

print("Processing files found in the directory...")

if 'items' not in locals():
    print("Error: The 'items' variable does not exist.")
    print("Please run the previous code cell (the one with pathlib) first.")
else:
    for p in items:
        # We only care about files, not directories
        if not p.is_file():
            continue

        filename = p.name # Get the filename string from the Path object
        
        # Filter for the tweet CSVs
        if not filename.startswith('tweets_') or not filename.endswith('.csv'):
            continue
            
        # --- 1. Read row count (Amount of Tweets) ---
        # --- !! MODIFICATION !! ---
        # We will count tweets by summing chunk lengths from pd.read_csv
        # This is more accurate than line counting if tweets contain newlines.
        tweet_count = 0 
        
        # --- 2. Read Date Range (from created_at column) ---
        date_range_str = "N/A"
        duration_str = "N/A"
        min_date_dt = None
        max_date_dt = None
        
        # --- !! MODIFICATION HERE !! ---
        # Define the expected date format to speed up parsing and remove warnings.
        # Assuming a common format like '2025-10-20 17:48:58'
        # If your format is different (e.g., has timezone), you may need to adjust this.
        # For ISO format with timezone (e.g., 2025-10-20T17:48:58.000Z), you might not need format='mixed'.
        # We will try 'mixed' first as a robust way to handle multiple fast formats.
        twitter_date_format = "mixed" 
        
        try:
            # Use iterator to read in chunks for memory efficiency
            chunk_iter = pd.read_csv(
                p, 
                usecols=['created_at'], 
                parse_dates=['created_at'],
                # Add the format parameter here
                date_format=twitter_date_format, 
                chunksize=100000,
                lineterminator='\n' # Handle potential malformed lines
            )
            
            for chunk in chunk_iter:
                # --- !! MODIFICATION !! ---
                tweet_count += len(chunk) # Add chunk length to total count
                
                if chunk.empty:
                    continue
                
                chunk_valid_dates = chunk['created_at'].dropna()
                if chunk_valid_dates.empty:
                    continue
                    
                chunk_min = chunk_valid_dates.min()
                chunk_max = chunk_valid_dates.max()
                
                if min_date_dt is None or chunk_min < min_date_dt:
                    min_date_dt = chunk_min
                if max_date_dt is None or chunk_max > max_date_dt:
                    max_date_dt = chunk_max
                    
        except Exception as e:
            # If the format guess was wrong, let's try again without it
            if "time data" in str(e) or "format" in str(e):
                print(f"Warning: Format '{twitter_date_format}' failed for {filename}. Retrying with infer_datetime_format...")
                tweet_count = 0 # Reset count for retry
                try:
                    chunk_iter = pd.read_csv(
                        p, 
                        usecols=['created_at'], 
                        parse_dates=['created_at'], 
                        infer_datetime_format=True, # Old way, but fallback
                        chunksize=100000,
                        lineterminator='\n'
                    )
                    for chunk in chunk_iter:
                        # --- !! MODIFICATION !! ---
                        tweet_count += len(chunk) # Add chunk length to total count

                        if chunk.empty:
                            continue
                        chunk_valid_dates = chunk['created_at'].dropna()
                        if chunk_valid_dates.empty:
                            continue
                        chunk_min = chunk_valid_dates.min()
                        chunk_max = chunk_valid_dates.max()
                        if min_date_dt is None or chunk_min < min_date_dt:
                            min_date_dt = chunk_min
                        if max_date_dt is None or chunk_max > max_date_dt:
                            max_date_dt = chunk_max
                except Exception as e2:
                    print(f"Error: Fallback parsing also failed for {filename}: {e2}")
                    date_range_str = "Read Error"
                    tweet_count = "Error" # Mark count as error

            elif "usecols" in str(e) or "created_at" in str(e):
                 date_range_str = "No 'created_at' column"
                 tweet_count = "Error" # Mark count as error
            else:
                date_range_str = "Read Error"
                print(f"Warning: Could not read date range for {filename}: {e}")
                tweet_count = "Error" # Mark count as error

        # --- 3. Classify Duration based on actual dates ---
        if min_date_dt and max_date_dt:
            date_range_str = f"{min_date_dt.strftime('%Y-%m-%d')} to {max_date_dt.strftime('%Y-%m-%d')}"
            
            start_period = get_duration_period(min_date_dt.date())
            end_period = get_duration_period(max_date_dt.date())
            
            if start_period == end_period:
                duration_str = start_period
            else:
                duration_str = f"{start_period} - {end_period}"
        elif min_date_dt is None and date_range_str == "N/A": # If no error, but no dates found
            date_range_str = "No Dates Found"

        # --- !! NEW STEP 3.5: Check for completeness !! ---
        completeness_check = "N/A"
        missing_data_to_scrape = "N/A"
        margin = timedelta(days=1)

        if min_date_dt and max_date_dt:
            min_date = min_date_dt.date()
            max_date = max_date_dt.date()
            missing_parts = []

            if duration_str == "Before Demo":
                is_complete = (min_date <= before_demo_start + margin) and (max_date >= before_demo_end - margin)
                completeness_check = "Complete" if is_complete else f"Incomplete (Expected: {before_demo_start} to {before_demo_end})"
                if not is_complete:
                    if min_date > before_demo_start + margin:
                        missing_parts.append(f"since:{before_demo_start} until:{min_date - timedelta(days=1)}")
                    if max_date < before_demo_end - margin:
                        missing_parts.append(f"since:{max_date + timedelta(days=1)} until:{before_demo_end}")
                        
            elif duration_str == "Demo":
                is_complete = (min_date <= demo_start + margin) and (max_date >= demo_end - margin)
                completeness_check = "Complete" if is_complete else f"Incomplete (Expected: {demo_start} to {demo_end})"
                if not is_complete:
                    if min_date > demo_start + margin:
                        missing_parts.append(f"since:{demo_start} until:{min_date - timedelta(days=1)}")
                    if max_date < demo_end - margin:
                        missing_parts.append(f"since:{max_date + timedelta(days=1)} until:{demo_end}")

            elif duration_str == "After Demo":
                is_complete = (min_date <= after_demo_start + margin) and (max_date >= after_demo_end - margin)
                completeness_check = "Complete" if is_complete else f"Incomplete (Expected: {after_demo_start} to {after_demo_end})"
                if not is_complete:
                    if min_date > after_demo_start + margin:
                        missing_parts.append(f"since:{after_demo_start} until:{min_date - timedelta(days=1)}")
                    if max_date < after_demo_end - margin:
                        missing_parts.append(f"since:{max_date + timedelta(days=1)} until:{after_demo_end}")

            elif " - " in duration_str: # Catches "Before Demo - Demo", etc.
                completeness_check = "Spans Multiple Periods"
            elif duration_str == "Outside Range":
                completeness_check = "N/A (Outside Range)"
            elif duration_str == "No Dates Found":
                completeness_check = "N/A (No Dates)"
            
            if missing_parts:
                missing_data_to_scrape = f"Scrape: {', '.join(missing_parts)}"
        
        # --- 4. Parse Filename for Keywords and Scrape Time ---
        keywords = "N/A"
        scrape_date = "N/A"
        scrape_time = "N/A"

        # Find scrape date and time from filename
        match = scrape_datetime_re.search(filename)
        if match:
            scrape_date = match.group(1)
            scrape_time = match.group(2)
            
            # Get the base name (everything before the scrape timestamp)
            base_name = filename[:match.start()]
            
            # Split at the first '_since_'
            if '_since_' in base_name:
                keyword_part, date_part = base_name.split('_since_', 1)
                keywords = keyword_part.removeprefix('tweets_').lower()
        # --- 5. Append all data ---
        parsed_data.append({
            "Filename": filename,
            "Keywords": keywords,
            "Duration (from data)": duration_str, # Use new duration
            "Completeness Check": completeness_check,
            "Missing Data to Scrape": missing_data_to_scrape,
            "Amount (Tweets)": tweet_count,
            "Date Range (from data)": date_range_str,
            "Scrape Date": scrape_date,
            "Scrape Time": scrape_time,
            "min_date_dt": min_date_dt,
            "max_date_dt": max_date_dt
        })

if not parsed_data:
    print("No 'tweets_*.csv' files were found or processed in the 'items' list.")
else:
    # Create DataFrame
    df = pd.DataFrame(parsed_data)
    
    # --- !! NEW !! Convert Amount to numeric for safe summing ---
    # This handles any 'Error' strings, converting them to 0
    df['Amount (Tweets)'] = pd.to_numeric(df['Amount (Tweets)'], errors='coerce').fillna(0).astype(int)
    
    # --- Sorting Logic ---
    # Create a helper column for proper datetime sorting
    # It will be dropped before final display
    df['Scrape_DateTime'] = pd.to_datetime(
        df['Scrape Date'] + df['Scrape Time'], 
        format='%Y%m%d%H%M%S',
        errors='coerce' # Handle any parsing errors gracefully
    )
    
    # Sort by the new Scrape_DateTime column (earliest to latest)
    df_sorted = df.sort_values(by='Scrape_DateTime').drop(columns=['Scrape_DateTime'])

    # --- Display the sorted DataFrame ---
    print("\n--- Parsed Tweet Metadata (Sorted by Scrape Time) ---")
    
    # Use pandas options to print the full table without truncating
    with pd.option_context('display.max_rows', None, 
                           'display.max_columns', None, 
                           'display.width', 1000):
        print(df_sorted.drop(columns=["min_date_dt", "max_date_dt"])) # Drop helper dates for display

    # --- Save to a new CSV file ---
    output_filename = "parsed_tweet_filenames_sorted.csv"
    df_sorted.drop(columns=["min_date_dt", "max_date_dt"]).to_csv(output_filename, index=False)
    print(f"\nSuccessfully saved sorted results to {output_filename}")


    # --- !! NEW SECTION: KEYWORD COMPLETENESS SUMMARY !! ---
    print("\n--- Keyword Completeness Summary ---")
    
    keyword_summary_data = []
    margin = timedelta(days=1)
    
    # Group by Keywords
    keyword_groups = df.groupby('Keywords')
    
    for keyword, group in keyword_groups:
        if keyword == "N/A":
            continue
            
        # Check Before Demo period
        before_stats = check_keyword_completeness(group, "Before Demo", before_demo_start, before_demo_end, margin)
        before_stats['Keyword'] = keyword
        
        # Check Demo period
        demo_stats = check_keyword_completeness(group, "Demo", demo_start, demo_end, margin)
        demo_stats['Keyword'] = keyword
        
        # Check After Demo period
        after_stats = check_keyword_completeness(group, "After Demo", after_demo_start, after_demo_end, margin)
        after_stats['Keyword'] = keyword
        
        keyword_summary_data.extend([before_stats, demo_stats, after_stats])

    # Create and display the summary DataFrame
    df_keyword_summary = pd.DataFrame(keyword_summary_data)
    
    # Reorder columns for clarity
    summary_cols = ['Keyword', 'Period', 'Amount (Tweets)', 'Completeness', 'Combined Date Range', 'Missing to Scrape'] # <-- 'Amount (Tweets)' ADDED
    df_keyword_summary = df_keyword_summary[summary_cols]

    with pd.option_context('display.max_rows', None, 
                           'display.max_columns', None, 
                           'display.width', 1000):
        print(df_keyword_summary)

    # --- Save summary to a new CSV file ---
    summary_output_filename = "keyword_completeness_summary.csv"
    df_keyword_summary.to_csv(summary_output_filename, index=False)
    print(f"\nSuccessfully saved keyword summary to {summary_output_filename}")

